In [30]:
from docx import Document
import pandas as pd

In [31]:

file_path = "AHR_data 29th.xlsx"
prev_file_path = "AHR_data 28th.xlsx"


all_sheets = pd.read_excel(file_path, sheet_name=None)
all_sheets_prev = pd.read_excel(prev_file_path, sheet_name=None)

state_data = {}

state_data_prev = {}


for sheet_name, df in all_sheets.items():
    df.columns = df.columns.str.strip()
    df = df[df['state'] != 'Total US'].copy()
    for _, row in df.iterrows():
        state = row['state']
        if state not in state_data:
            state_data[state] = {}
        state_data[state][sheet_name] = row.to_dict()
        
        

for sheet_name, df in all_sheets_prev.items():
    df.columns = df.columns.str.strip()
    df = df[df['state'] != 'Total US'].copy()
    for _, row in df.iterrows():
        state = row['state']
        if state not in state_data_prev:
            state_data_prev[state] = {}
        state_data_prev[state][sheet_name] = row.to_dict()
      
print("Current file sheets:", list(all_sheets.keys()))
print("Current file 'All Rankings' columns:", all_sheets["All Rankings"].columns.tolist())

print("Previous file sheets:", list(all_sheets_prev.keys()))
print("Previous file 'All Rankings' columns:", all_sheets_prev["All Rankings"].columns.tolist())

current_cols = set(all_sheets["All Rankings"].columns)
prev_cols = set(all_sheets_prev["All Rankings"].columns)
diff_cols = current_cols.symmetric_difference(prev_cols)
print("Differences in 'All Rankings' columns between current and previous files:", diff_cols)  
          


Current file sheets: ['AHR Data', 'Scores & Rankings', 'All Rankings', 'Disbursement Data']
Current file 'All Rankings' columns: ['year', 'state', 'other_fatalities_per_100m_VMT_rank', 'poor_bridges_percent_rank', 'rural_OPA_poor_percent_rank', 'rural_fatalities_per_100m_VMT_rank', 'rural_interstate_poor_percent_rank', 'state_avg_congestion_hours_rank', 'urban_OPA_poor_percent_rank', 'urban_fatalities_per_100m_VMT_rank', 'urban_interstate_poor_percent_rank', 'admin_disbursement_perlm_rank', 'capital_disbursement_perlm_rank', 'maintenance_disbursement_perlm_rank', 'other_disbursement_perlm_rank', 'overall_rank', 'state_avg_congestion_hours']
Previous file sheets: ['AHR Data', 'Scores & Rankings', 'All Rankings', 'Disbursement Data']
Previous file 'All Rankings' columns: ['year', 'state', 'other_fatalities_per_100m_VMT_rank', 'poor_bridges_percent_rank', 'rural_OPA_poor_percent_rank', 'rural_fatalities_per_100m_VMT_rank', 'rural_interstate_poor_percent_rank', 'state_avg_congestion_hours_

In [32]:
metric_label_map = {
    "overall_rank": "Overall",
    "capital_disbursement_perlm_rank": "Capital & Bridge Disbursements",
    "maintenance_disbursement_perlm_rank": "Maintenance Disbursements",
    "admin_disbursement_perlm_rank": "Administrative Disbursements",
    "other_disbursement_perlm_rank": "Other Disbursements",
    "rural_interstate_poor_percent_rank": "Rural Interstate Pavement Condition",
    "rural_OPA_poor_percent_rank": "Rural Other Principal Arterial Pavement Condition",
    "urban_interstate_poor_percent_rank": "Urban Interstate Pavement Condition",
    "urban_OPA_poor_percent_rank": "Urban Other Principal Arterial Pavement Condition",
    "state_avg_congestion_hours_rank": "Urbanized Area Congestion",
    "poor_bridges_percent_rank": "Structurally Deficient Bridges",
    "rural_fatalities_per_100m_VMT_rank": "Rural Fatality Rate",
    "urban_fatalities_per_100m_VMT_rank": "Urban Fatality Rate",
    "other_fatalities_per_100m_VMT_rank": "Other Fatality Rate"
}


def get_metric_label(metric_key):
        return metric_label_map.get(metric_key, metric_key.replace('_rank', '').replace('_', ' ').title())

In [45]:

import os
import json

with open("state_comparisons_clean.json", "r") as f:
    state_comparisons = json.load(f)
    #print(list(state_comparisons.keys()))


def safe_ordinal(n):
    if pd.isna(n):
        return "unranked"
    return str(int(n))

def ordinal(n):
    return f"{int(n)}" + ("th" if 11 <= int(n) % 100 <= 13 else {1: "st", 2: "nd", 3: "rd"}.get(int(n) % 10, "th"))

def generate_report(state, data, prev_data):
    doc = Document()

    score_data = data['All Rankings']
    prev_score_data = prev_data['All Rankings']

    overall_rank = safe_ordinal(score_data['overall_rank'])
    congestion_rank = safe_ordinal(score_data['state_avg_congestion_hours_rank'])
    congestion_hours = round(score_data['state_avg_congestion_hours'], 2)

    doc.add_heading(f"{state} Ranks {overall_rank} in the Nation in Highway Performance and Cost-Effectiveness", level=1)

    doc.add_paragraph(
        f"{state}’s highway system ranks {overall_rank} in the nation in overall cost-effectiveness and condition."
    )

    doc.add_paragraph(
        f"According to the Annual Highway Report by Reason Foundation, this is a change from the state's previous rankings. "
    )

    doc.add_paragraph(
        f"In safety and condition categories, {state}’s highways rank "
        f"{safe_ordinal(score_data['urban_interstate_poor_percent_rank'])} in urban Interstate pavement condition, "
        f"{safe_ordinal(score_data['rural_interstate_poor_percent_rank'])} in rural Interstate pavement condition, "
        f"{safe_ordinal(score_data['urban_OPA_poor_percent_rank'])} in urban arterial pavement condition, "
        f"{safe_ordinal(score_data['rural_OPA_poor_percent_rank'])} in rural arterial pavement condition, "
        f"{safe_ordinal(score_data['poor_bridges_percent_rank'])} in structurally deficient bridges, "
        f"{safe_ordinal(score_data['urban_fatalities_per_100m_VMT_rank'])} in urban fatality rate, and "
        f"{safe_ordinal(score_data['rural_fatalities_per_100m_VMT_rank'])} in rural fatality rate."
    )

    doc.add_paragraph(
        f"{state} ranks {congestion_rank} out of the 50 states in traffic congestion, and its drivers spend {congestion_hours} hours a year stuck in traffic congestion."
    )

    doc.add_paragraph(
        f"In spending and cost-effectiveness, {state} ranks "
        f"{safe_ordinal(score_data['capital_disbursement_perlm_rank'])} in capital and bridge disbursements, which are the costs of building new roads and bridges and widening existing ones. "
        f"{state} ranks {safe_ordinal(score_data['maintenance_disbursement_perlm_rank'])} in maintenance spending, such as the costs of repaving roads and filling in potholes. "
        f"{state}’s administrative disbursements, including office spending that doesn’t make its way to roads, rank {safe_ordinal(score_data['admin_disbursement_perlm_rank'])} nationwide."
    )

    changes = []
    for key in score_data:
        if key.endswith("_rank") and key in prev_score_data:
            prev_val = prev_score_data[key]
            curr_val = score_data[key]
            if isinstance(prev_val, (int, float)) and isinstance(curr_val, (int, float)):
                diff = prev_val - curr_val
                if abs(diff) >= 5:
                    direction = "improved" if diff > 0 else "worsened"
                    changes.append(
                        f"{get_metric_label(key)} {direction} by {abs(diff)} points, from {int(prev_val)} to {int(curr_val)}"
                    )

    if changes:
        doc.add_paragraph(f"Compared to last year, {state} has the following notable changes in rankings: " + "; ".join(changes) + ".")
    else:
        doc.add_paragraph(f"There are no significant changes in rankings (5 or more places) for {state} compared to last year.")    
                

    comparison_data = state_comparisons.get(state, {})
    neighbors = comparison_data.get("neighboring_states", [])
    similar_states = comparison_data.get("similarly_populated_states", [])
    
    def get_state_rank(s):
        try:
            return int(all_sheets["All Rankings"].loc[all_sheets["All Rankings"]["state"] == s, "overall_rank"].values[0])
        except (IndexError, KeyError):
            return None

    def describe_comparison(state_list, reference_rank):
        comparison = []
        for s in state_list:
            rank = get_state_rank(s)
            if rank is not None:
                direction = "better than" if rank > reference_rank else "worse than"
                comparison.append((direction, f"{s} ({safe_ordinal(rank)})"))
        if not comparison:
            return None
        grouped = {}
        for direction, text in comparison:
            grouped.setdefault(direction, []).append(text)
        parts = []
        for direction, states in grouped.items():
            joined = (
                ", ".join(states[:-1]) + (", and " if len(states) > 2 else " and ") + states[-1]
                if len(states) > 1 else states[0]
            )
            parts.append(f"{direction} {joined}")
        return "; ".join(parts)

    overall_rank_value = score_data.get("overall_rank")
    neighbor_sentence = ""
    similar_sentence = ""

    if isinstance(overall_rank_value, (int, float)):
        neighbor_desc = describe_comparison(neighbors, overall_rank_value)
        similar_desc = describe_comparison(similar_states, overall_rank_value)
        if neighbor_desc:
            if state == "Alaska":
                neighbor_sentence = f"Compared to other somewhat similar states, {state}’s overall highway performance is {neighbor_desc}."
            else:
                neighbor_sentence = f"Compared to neighboring and nearby states, {state}’s overall highway performance is {neighbor_desc}."
                
        if similar_desc:
            similar_sentence = f"Comparing its overall performance to similarly populated states, {state} ranks {similar_desc}."

    if neighbor_sentence:
        doc.add_paragraph(neighbor_sentence)
    if similar_sentence:
        doc.add_paragraph(similar_sentence)
    

    

    prev_rank = prev_score_data.get('overall_rank')
    trend_text = (
        f"{state}’s highway system ranks {overall_rank} out of 50 states overall this year, "
        f"compared to {safe_ordinal(prev_rank)} last year."
    ) if isinstance(prev_rank, (int, float)) else (
        f"{state}’s highway system ranks {overall_rank} out of 50 states overall this year."
    )
    doc.add_paragraph(trend_text)

    # Identify the two worst metrics (highest ranks) excluding NaNs
    ranked_metrics = {
        k: v for k, v in score_data.items() if k.endswith("_rank") and pd.notna(v)
    }
    worst_two = sorted(ranked_metrics.items(), key=lambda item: item[1], reverse=True)[:2]
    worst_metric_1 = get_metric_label(worst_two[0][0])
    worst_metric_2 = get_metric_label(worst_two[1][0])

    if any(score_data[m] >= 40 for m in ranked_metrics):
        worst_metric_value = score_data[worst_two[0][0]]
        doc.add_paragraph(
            f'“In terms of improving in the road condition and performance categories, {state} should focus on reducing {worst_metric_1} and {worst_metric_2}. '
            f'{state}’s rank of {safe_ordinal(worst_metric_value)} in {get_metric_label(worst_two[0][0])} makes it one of the worst in the nation for this safety metric,” '        
            f'said Baruch Feigenbaum, lead author of the 29th Annual Highway Report and senior managing director of transportation policy at Reason Foundation.'
    )
    elif all(score_data[m] <= 10 for m in ranked_metrics):
        doc.add_paragraph(
            f'“While {state} ranks in the top 10 of 50 states overall, the state should still look to improve its performance in {worst_metric_1} and {worst_metric_2}, its two weakest metrics,” said Baruch Feigenbaum, lead author of the 29th Annual Highway Report and senior managing director of transportation policy at Reason Foundation.'
        )
    else:
        doc.add_paragraph(
            f'“{state} should focus on reducing {worst_metric_1} and {worst_metric_2}, its two weakest metrics,” said Baruch Feigenbaum, lead author of the 28th Annual Highway Report and senior managing director of transportation policy at Reason Foundation.'
        )
        
    # Add table of rankings
    doc.add_paragraph(f"{state.upper()}’S RANKINGS IN THE 29TH ANNUAL HIGHWAY REPORT", style="Heading 2")
    
    # Create table with header row
    table = doc.add_table(rows=1, cols=2)
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'Category'
    hdr_cells[1].text = 'Rank'
    
    # Fill in rankings from score_data using metric_label_map
    for metric_key, label in metric_label_map.items():
        value = score_data.get(metric_key)
        if pd.notna(value):
            row_cells = table.add_row().cells
            row_cells[0].text = label
            row_cells[1].text = safe_ordinal(value)    
        
        
    doc.add_paragraph(
        f"\nReason Foundation’s 29th Annual Highway Report measures the condition and cost- effectiveness of state-controlled highways in 13 categories, including pavement and bridge conditions, traffic fatalities, and spending." 
        f" In the performance categories, ranking first implies the state has the best or lowest fatality rate and its road pavement is in the best condition."
        f" A ranking of 50th in performance categories means the state has the worst fatality rates or pavement conditions."
        f" In simplified terms, in the cost-effectiveness categories, a rank of 50 means the state spends more money, and a first-place ranking means the state spends less money than other states in that category."
    )
    
    doc.add_paragraph(
        f"The report’s data are primarily information each state directly reported to the Federal Highway Administration for 2023. "
        f"Better Roads and Bridges provides the deficient bridge data, and INRIX.com provides the traffic congestion data."
    )

    filename = f"{state.lower().replace(' ', '_')}_highway_report.docx"
    doc.save(os.path.join("state-vignettes", filename))


In [46]:

for state in ["Alaska"]:
        generate_report(state, state_data[state], state_data_prev[state])